# Task 10 — Model Evaluation and Optimization

Use a per-user holdout: the last-listed positive interaction (rating ≥ 4) is hidden for each eligible user. Metrics are Precision@K and Recall@K. With one held-out relevant talk per user, Recall@K is hit rate; results are a small offline diagnostic, not a reliable production estimate.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "dataset").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.recommender import TEDRecommender
model = TEDRecommender.from_csv(ROOT / "dataset/ted_talks_cleaned.csv", ROOT / "dataset/user_interactions.csv")
print(f"Loaded {len(model.talks):,} talks and {len(model.interactions):,} interactions across {len(model.user_item):,} users")

## Metrics and held-out comparison

In [ ]:
from src.recommender import precision_recall_at_k
# Keep one positive target per user, and train on all their remaining interactions.
positive = model.interactions[model.interactions.rating >= 4].copy()
positive["order"] = positive.index
test = positive.sort_values("order").groupby("user_id").tail(1)[["user_id","talk_id"]]
train = model.interactions.merge(test.assign(_heldout=1), on=["user_id","talk_id"], how="left")
train = train[train._heldout.isna()].drop(columns="_heldout")
eval_model = TEDRecommender(model.talks, train)
results = []
for weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
    metric_lists = {name:[] for name in ["Content", "Collaborative", "Hybrid"]}
    for row in test.itertuples(index=False):
        uid, target = str(row.user_id), str(row.talk_id)
        recs = {"Content":eval_model.content_recommendations(uid, top_n=10),
                "Collaborative":eval_model.collaborative_recommendations(uid, top_n=10),
                "Hybrid":eval_model.hybrid_recommendations(uid, top_n=10, content_weight=weight)}
        for name, frame in recs.items():
            p,r = precision_recall_at_k(frame.talk_id.tolist(), [target], k=10)
            metric_lists[name].append((p,r))
    for name, vals in metric_lists.items():
        results.append({"content_weight":weight if name=="Hybrid" else None,"model":name,
                        "Precision@10":sum(x[0] for x in vals)/max(1,len(vals)),
                        "Recall@10":sum(x[1] for x in vals)/max(1,len(vals)),"users":len(vals)})
evaluation = pd.DataFrame(results)
display(evaluation)

## Model comparison and limitations

In [ ]:
comparison = evaluation.groupby("model")[["Precision@10","Recall@10"]].mean().reset_index()
display(comparison)

Precision@K is the share of the top K items that are relevant; Recall@K is the share of all relevant held-out items that appear in the top K. Here only one positive is held out per user, and only 30 users exist, so both metrics are noisy and the recall definition is constrained. The file has no event timestamp, so row order is only a reproducible proxy and not a true temporal split. A timestamped repeated temporal split and stronger relevance data are needed for a confident final decision.

**Strengths:** content can recommend for new users with a seed talk; collaborative filtering can surface preference patterns not apparent from text; hybrid covers both signals. **Limitations:** sparse interactions, cold-start users/items, lexical TF-IDF limitations, and an artificial/small interaction sample.

**Final configuration:** start with a 0.5 content weight as a transparent baseline; inspect the table above and adjust only if a weight improves both metrics consistently. This is a provisional configuration because the available sample is too small for a confident winner.